In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import dgl
import random
from scipy.sparse import csr_matrix
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics.cluster import normalized_mutual_info_score
import os

In [6]:
import so as so

In [7]:
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
dgl.random.seed(seed)

In [8]:
sc.set_figure_params(dpi=150, figsize=(2, 2), frameon=False)    # TODO 是否画边框

In [9]:
files = os.listdir('/dataset/3_large_wang_zeng/')
files

['mouse_BrainAtals_well11_WangXiao.h5ad',
 'mouse_brain_C57BL6J-2.061_zhuang.h5ad',
 'mouse_BrainAtals_sagittal1_WangXiao.h5ad',
 'mouse_brainC57BL6J-638850.37_zeng.h5ad']

In [10]:
files = [
    'mouse_BrainAtals_well11_WangXiao.h5ad',
 'mouse_brain_C57BL6J-2.061_zhuang.h5ad',
 'mouse_BrainAtals_sagittal1_WangXiao.h5ad',
 'mouse_brainC57BL6J-638850.37_zeng.h5ad',
]

# load data

In [11]:
data_list = [sc.read_h5ad(f'/dataset/3_large_wang_zeng/{file}') for file in files]
data_list

[AnnData object with n_obs × n_vars = 43341 × 1022
     obs: 'Main_molecular_cell_type', 'Sub_molecular_cell_type', 'Main_molecular_tissue_region', 'Sub_molecular_tissue_region', 'Molecular_spatial_cell_type', 'subregion'
     uns: 'Main_molecular_tissue_region_colors', 'subregion_colors'
     obsm: 'spatial',
 AnnData object with n_obs × n_vars = 28240 × 1120
     obs: 'brain_section_label', 'major_brain_region', 'ccf_region_name', 'cell_type', 'subclass_transfer', 'subregion'
     var: 'gene_name', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'ENS'
     uns: 'ccf_region_name_colors', 'citation', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'subregion_colors', 'title'
     obsm: 'X_CCF', 'X_spatial_coords', 'X_umap', 'spatial',
 AnnData object with n_obs × n_vars = 91246 × 1022
     obs: 'Main_molecular_cell_type', 'Sub_molecular_cell_type', 'Main_molecular_tissue_region', 'Sub_molecu

In [12]:
batch_names = files
# data_list[1].var.index = [item.upper() for item in data_list[1].var.index]
# data_list[1].obsm['spatial'] = data_list[1].obsm['sptial_section'][:, 0: 2]
batch_names

['mouse_BrainAtals_well11_WangXiao.h5ad',
 'mouse_brain_C57BL6J-2.061_zhuang.h5ad',
 'mouse_BrainAtals_sagittal1_WangXiao.h5ad',
 'mouse_brainC57BL6J-638850.37_zeng.h5ad']

In [13]:
import numpy as np
np.set_printoptions(suppress=True)

In [14]:
batch_key = 'batch'
adata = so.pp.process_adata(data_list, label=batch_key, keys=batch_names, n_top_features=None, target_sum=1e4)
adata

AnnData object with n_obs × n_vars = 252972 × 258
    obs: 'subregion', 'original_index', 'spot_quality', 'batch'
    uns: 'log1p', 'mu_bg', 'std_bg'
    obsm: 'spatial'
    layers: 'counts', 'norm_log'

In [15]:
adata.X.max(), adata.X.min()

(9.210440366976515, 0.0)

In [16]:
rad_cutoff_list = [300, 70, 300, 0.07]
adata, g = so.pp.process_graph(adata, data_list, cutoff_list=rad_cutoff_list)

cal spatial net in data_list
------Calculating spatial graph...
The graph contains 863482 edges, 43341 cells.
19.9230 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 954602 edges, 28240 cells.
33.8032 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 2988398 edges, 91246 cells.
32.7510 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 2944689 edges, 90145 cells.
32.6661 neighbors per cell on average.


In [17]:
path_results = f'./log/integration_radius_256_4datasets/'

In [18]:
adata

AnnData object with n_obs × n_vars = 252972 × 258
    obs: 'subregion', 'original_index', 'spot_quality', 'batch'
    uns: 'log1p', 'mu_bg', 'std_bg', 'adj'
    obsm: 'spatial'
    layers: 'counts', 'norm_log'

In [19]:
adata_int = so.model.SpaVCCA(adata=adata,
                            n_epoch=5,
                            batch_size=1024,
                            verbose=False,
                            gpu=0,
                            random_seed=42,
                            output_dir=path_results,
                            num_heads=2,
                            h_dim=16,
                            reconstruct_loss='orig',
                            alpha_kl=1,
                            alpha_re=1,
                            alpha_mmd=1,
                            alpha_spl=1,
                            )

clip
reconstruct_loss == orig


Epochs: 100%|█| 5/5 [01:03<00:00, 12.74s/it, kl_loss=5.1557, recon_loss=516.4830, cca_loss=0.0061, sub_par_loss=0.3488, 
Infer latent: 100%|███████████████████████████████████████████████████████████████████| 248/248 [00:04<00:00, 54.54it/s]


In [21]:
del adata_int.uns['adj']

In [22]:
adata_int.write_h5ad(f'{path_results}adata_int.h5ad', compression='gzip')